# Unified Privacy Audit Runner — DRO × DP-SGD × MIA
## Self-Tuning ε-Sweep with Pre-Clip Gradient Logging & Paired Statistics

**Run one instance per dataset.** Set `DATASET` in Cell 1 (or via env var).

### What's new — the self-improving auto-tuner

The notebook now runs up to `max_rounds` sweep passes. After each pass it
evaluates every (ε, method) cell against a quality rubric (C1 fairness,
C2 privacy leakage, C3 disparate vulnerability, DRO activation, signal
presence, subgroup sample size). Failed cells trigger parameter adjustments
(α, max_ratio, uniform_mix, k, lr, warmup) according to six diagnostic rules.
Only failed cells are re-run; passing cells are kept via the checkpoint logic.

| Failure pattern                                  | Auto-adjustment                                |
|--------------------------------------------------|------------------------------------------------|
| λ/max < 10% (DRO dormant)                        | α ← α × 0.4                                    |
| λ/max > 90% + utility diverging (saturated)     | α ← α × 2.0                                    |
| DRO active but race_gap not closing (C1 fails)   | max_ratio × 2, uniform_mix × 0.5                |
| DRO active but all AUCs ≈ 0.5 (no signal) & k<2k | k ← k × 2 (max 2000)                           |
| Any subgroup n_in or n_out < 30                  | k ← k × 2 (max 2000)                           |
| Trivial collapse (BCE ≈ ln 2)                    | lr × 0.5, dro_warmup + 3                       |

### Original pipeline (per dataset × ε × method)
1. Train ERM+DP-SGD and χ²-DRO+DP-SGD on identical split + canaries
2. Black-box attack (loss drop) + White-box attack (gradient alignment)
3. Log **pre-clip gradient norms** per canary → subgroup mechanistic evidence
4. Steinke one-run audit → ε_lb, AUC, V, p
5. Paired Wilcoxon + Cohen's d (ERM vs DRO, same canaries)
6. Smart checkpoint: resume on crash; auto-validate DRO activation

Formulas implemented exactly per:
- Hashimoto et al. (2018) Alg. 1 — χ²-DRO online mirror descent
- Abadi et al. (2016) — DP-SGD
- Steinke, Nasr & Jagielski (2023) — one-run auditing


In [1]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit here or set env vars before launching Jupyter
# ══════════════════════════════════════════════════════════════════════
import os
from pathlib import Path

DATASET = os.environ.get("DATASET", "adult")          # 'texas' | 'adult' | 'utkface'
GPU_INDEX = int(os.environ.get("GPU", "0"))

# ε sweep: None = No-DP baseline; others are TARGET epsilons (σ auto-mapped, realised ε reported)
EPSILON_SWEEP = [None, 8.0, 4.0, 2.0]

# Paths — edit MONOLITH_ROOT / DATA_ROOT to match your machine
PROJECT_ROOT = Path.home() / "project"
MONOLITH_ROOT = PROJECT_ROOT / "monolith"
DATA_ROOT     = MONOLITH_ROOT / "data"
EXP_ROOT      = MONOLITH_ROOT / "experiments" / f"{DATASET}_eps_sweep"
EXP_ROOT.mkdir(parents=True, exist_ok=True)

SEED, AUDIT_SEED = 2024, 42

# Auto-tuner budget
MAX_TUNING_ROUNDS = 3

# Per-dataset hyperparameters (α auto-calibrated in Cell 13; these are fallbacks/initials)
DATASET_CFG = {
    "utkface": dict(
        task="regression", model="resnet18", epochs=25, batch_size=64, phys_batch=32,
        lr=1e-4, weight_decay=1e-4, val_frac=0.15, k=500,
        dro_alpha_init=150.0, dro_eta=0.001, dro_lambda_init=0.0005, dro_warmup=2,
        sub_epoch_frac=0.8, sigma_map={8.0: 1.1, 4.0: 2.0, 2.0: 4.0},
    ),
    "texas": dict(
        task="classification", model="mlp", epochs=60, batch_size=256, phys_batch=128,
        lr=3e-3, weight_decay=1e-4, val_frac=0.15, k=500,
        dro_alpha_frac=0.7,          # α = 0.7 × CONVERGED ERM variance (replaces dro_alpha_init)
        dro_eta=0.2, dro_lambda_init=0.0, dro_warmup=3,
        min_epochs=25, patience=12,  # give DRO a real window
        sub_epoch_frac=0.8, sigma_map={8.0: 1.0, 4.0: 2.0, 2.0: 4.0},
    ),
    "adult": dict(
        task="classification", model="mlp", epochs=60, batch_size=256, phys_batch=128,
        lr=3e-3, weight_decay=1e-4, val_frac=0.15, k=500,
        dro_alpha_frac=0.7,
        dro_eta=0.2, dro_lambda_init=0.0, dro_warmup=3,
        min_epochs=25, patience=12,
        sub_epoch_frac=0.8, sigma_map={8.0: 1.0, 4.0: 2.0, 2.0: 4.0},
    ),
}
CFG = DATASET_CFG[DATASET]

# Hard caps for the auto-tuner
MAX_K_HARD_CAP      = 2000      # don't let k grow unboundedly
MAX_ALPHA_RATIO     = 4.0       # don't change α by more than 4x in a single step
MAX_LR_RATIO        = 4.0       # don't change lr by more than 4x in a single step
MAX_WARMUP          = 15        # don't let warmup grow beyond 15 epochs
ALPHA_FLOOR = 0.05        # tabular BCE; prevents λ blow-up      
DP_MAX_GRAD_NORM = 1.0
DP_DELTA         = 1e-5
AUDIT_GAMMA      = 0.05

print(f"Dataset: {DATASET} | Task: {CFG['task']} | ε sweep: {EPSILON_SWEEP}")
print(f"Experiments → {EXP_ROOT}")
print(f"Auto-tune budget: {MAX_TUNING_ROUNDS} rounds")


Dataset: adult | Task: classification | ε sweep: [None, 8.0, 4.0, 2.0]
Experiments → /home/govindu.srimaan/project/monolith/experiments/adult_eps_sweep
Auto-tune budget: 3 rounds


In [2]:
import sys, gc, time, json, math, random, warnings
from datetime import datetime
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torchvision import transforms
from PIL import Image
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

device = torch.device(f"cuda:{GPU_INDEX}" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(device)
    print(f"Device: {device} — {torch.cuda.get_device_name(device)}")
else:
    print("⚠️ CPU only — this will be extremely slow for UTKFace.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# NOTE: Texas labels "Hispanic" → code 3 → "Indian" in RACE_MAP. The codes are
# arbitrary; the only requirement is consistency between training and subgroup
# analysis. If you want honest labels in the table, replace "Indian" with
# "Hispanic" when DATASET == 'texas'.
# Dataset-aware labels. Texas code 3 is Hispanic, NOT Indian.
_SUBGROUP_LABELS = {
    "utkface": {"race": {0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Others"}},
    "texas":   {"race": {0: "Caucasian", 1: "AfricanAmerican", 2: "Asian", 3: "Hispanic", 4: "Other"}},
    "adult":   {"race": {0: "White", 1: "Black", 2: "Asian-Pac-Islander",
                         3: "Amer-Indian-Eskimo", 4: "Other"}},
}
RACE_MAP   = _SUBGROUP_LABELS[DATASET]["race"]
GENDER_MAP = {0: "Male", 1: "Female"}

# Module-level override dict — set by the auto-tuner; read by run_cell.
# Empty dict = use CFG defaults (i.e., outside auto-tune mode).
_ACTIVE_OVERRIDES = {}


Device: cuda:0 — NVIDIA L40S


In [3]:
# ══════════════════════════════════════════════════════════════════════
# DATA LOADERS — hardened: FileNotFoundError instead of synthetic fallback
# ══════════════════════════════════════════════════════════════════════
import pandas as pd

class UTKFaceDataset(Dataset):
    """Robustly finds UTKFace images. Tries multiple layouts; raises if none."""
    def __init__(self, root_dir, transform=None):
        self.root_dir, self.transform, self.samples = Path(root_dir), transform, []

        candidate_dirs = [
            self.root_dir / "UTKFace",
            self.root_dir / "extracted" / "Privacy Auditing" / "UTKFace",
            self.root_dir / "extracted" / "UTKFace",
            self.root_dir,
        ]
        img_dir = next((d for d in candidate_dirs if d.exists() and any(d.rglob("*.jpg"))), None)
        if img_dir is None:
            raise FileNotFoundError(
                f"No UTKFace .jpg images found under any of: {[str(d) for d in candidate_dirs]}"
            )

        for p in sorted(img_dir.rglob("*.jpg")):
            try:
                parts = p.stem.replace(".chip", "").split("_")
                if len(parts) >= 4:
                    age, gender, race = int(parts[0]), int(parts[1]), int(parts[2])
                    if 1 <= age <= 116 and gender in (0, 1) and race in RACE_MAP:
                        self.samples.append({"path": str(p), "age": age, "gender": gender, "race": race})
            except Exception:
                pass
        if len(self.samples) == 0:
            raise FileNotFoundError(f"No valid UTKFace images parsed under {img_dir}")
        print(f"[UTKFace] Found {len(self.samples)} images under {img_dir}")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        return (img, torch.tensor(s["age"], dtype=torch.float32),
                torch.tensor(s["gender"], dtype=torch.long),
                torch.tensor(s["race"], dtype=torch.long))


class TensorDataset4(Dataset):
    """Tabular dataset returning (features, label, gender, race)."""
    def __init__(self, feats, labels, genders, races):
        self.f = torch.as_tensor(feats, dtype=torch.float32)
        self.l = torch.as_tensor(labels, dtype=torch.float32)
        self.g = torch.as_tensor(genders, dtype=torch.long)
        self.r = torch.as_tensor(races, dtype=torch.long)
    def __len__(self): return len(self.l)
    def __getitem__(self, i): return self.f[i], self.l[i], self.g[i], self.r[i]

class TabularLoader:
    """Loads Texas or Adult from the REAL csv. Raises if missing — never synthetic.
    Protected attributes (race, sex) are EXCLUDED from features and kept only as
    subgroup labels — this is what makes it a 'fairness without demographics' setup."""

    def __init__(self, dataset_name: str, data_dir: Path):
        if dataset_name == "texas":
            self._load_texas(data_dir)
        elif dataset_name == "adult":
            self._load_adult(data_dir)
        else:
            raise ValueError(dataset_name)

    # ── helpers ────────────────────────────────────────────────────────
    @staticmethod
    def _diag_cat(code) -> str:
        s = str(code).strip()
        if s in ("?", "nan", ""): return "missing"
        if s.startswith("V"): return "V"
        if s.startswith("E"): return "E"
        try: v = float(s)
        except ValueError: return "missing"
        if 390 <= v < 460 or v == 785: return "circulatory"
        if 460 <= v < 520 or v == 786: return "respiratory"
        if 520 <= v < 580 or v == 787: return "digestive"
        if 250 <= v < 251:            return "diabetes"
        if 800 <= v < 1000:           return "injury"
        if 140 <= v < 240:            return "neoplasms"
        if 580 <= v < 630 or v == 788: return "genitourinary"
        if 240 <= v < 280 or v == 790: return "endocrine"
        if 680 <= v < 710 or v == 782: return "skin"
        if 710 <= v < 740:            return "musculoskeletal"
        if v in (780, 781, 784) or 280 <= v < 290: return "nervous_blood"
        return "other"

    _AGE_BINS = {'[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4,
                 '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9}

    _MED_COLS = ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
                 'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
                 'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
                 'examide','citoglipton','insulin','glyburide-metformin',
                 'glipizide-metformin','glimepiride-pioglitazone',
                 'metformin-rosiglitazone','metformin-pioglitazone']

    def _finalize(self, feats, labels, races, genders, name):
        self.features = np.ascontiguousarray(feats, dtype=np.float32)
        self.labels   = np.asarray(labels,  dtype=np.float32)
        self.races    = np.asarray(races,   dtype=np.int64)
        self.genders  = np.asarray(genders, dtype=np.int64)
        self.input_dim = self.features.shape[1]
        print(f"[{name}] N={len(self.labels)} features={self.input_dim} "
              f"pos_rate={self.labels.mean():.3f}")
        for rv in range(5):
            print(f"    race {rv}: n={int((self.races==rv).sum()):6d} "
                  f"pos_rate={self.labels[self.races==rv].mean():.3f}")

    # ── Texas ──────────────────────────────────────────────────────────
    def _load_texas(self, data_dir):
        p = data_dir / "Texas" / "texas_hospital.csv"
        if not p.exists():
            raise FileNotFoundError(f"CRITICAL: real data not found at {p}")
        df = pd.read_csv(p, na_values=["?"], low_memory=False)

        # BALANCED LABEL: any readmission (>30 or <30) vs NO
        df["label"] = (df["readmitted"].astype(str).str.strip() != "NO").astype(int)

        race_map = {"Caucasian": 0, "AfricanAmerican": 1, "Asian": 2, "Hispanic": 3, "Other": 4}
        df["race_code"]   = df["race"].map(race_map).fillna(-1).astype(int)
        df["gender_code"] = df["gender"].map({"Male": 0, "Female": 1}).fillna(-1).astype(int)
        df = df[(df["race_code"] >= 0) & (df["gender_code"] >= 0)].reset_index(drop=True)

        num_cols = ["time_in_hospital","num_lab_procedures","num_procedures",
                    "num_medications","number_outpatient","number_emergency",
                    "number_inpatient","number_diagnoses"]
        num = df[num_cols].apply(pd.to_numeric, errors="coerce")
        num["age_bin"] = df["age"].map(self._AGE_BINS).astype(float)
        num = num.fillna(num.median())
        num = (num - num.mean()) / (num.std() + 1e-8)

        for c in ["diag_1","diag_2","diag_3"]:
            df[f"{c}_cat"] = df[c].map(self._diag_cat)

        cat_cols = ["admission_type_id","discharge_disposition_id","admission_source_id",
                    "max_glu_serum","A1Cresult","change","diabetesMed",
                    "diag_1_cat","diag_2_cat","diag_3_cat"]
        for c in cat_cols:
            df[c] = df[c].fillna("Unknown").astype(str)

        med_frames = []
        for c in self._MED_COLS:
            d = pd.get_dummies(df[c].astype(str), prefix=c)
            keep = [col for col in d.columns
                    if col.endswith(("_Steady","_Up","_Down"))]
            if keep: med_frames.append(d[keep].astype(np.float32))

        onehot = pd.get_dummies(df[cat_cols], columns=cat_cols).astype(np.float32)
        feats = np.hstack([num.values.astype(np.float32), onehot.values]
                          + [m.values for m in med_frames]).astype(np.float32)

        self._finalize(feats, df["label"], df["race_code"], df["gender_code"], "texas")

    # ── Adult ──────────────────────────────────────────────────────────
    def _load_adult(self, data_dir):
        p = data_dir / "Adult" / "adult.csv"
        if not p.exists():
            raise FileNotFoundError(f"CRITICAL: real data not found at {p}")
        df = pd.read_csv(p, na_values=["?"], skipinitialspace=True, low_memory=False)

        df["label"] = (df["income"].astype(str).str.strip() == ">50K").astype(int)
        race_map = {"White": 0, "Black": 1, "Asian-Pac-Islander": 2,
                    "Amer-Indian-Eskimo": 3, "Other": 4}
        df["race_code"]   = df["race"].map(race_map).fillna(-1).astype(int)
        df["gender_code"] = df["sex"].map({"Male": 0, "Female": 1}).fillna(-1).astype(int)
        df = df[(df["race_code"] >= 0) & (df["gender_code"] >= 0)].reset_index(drop=True)

        num_cols = ["age","fnlwgt","education-num","capital-gain","capital-loss","hours-per-week"]
        num = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        for c in ["fnlwgt","capital-gain","capital-loss"]:
            num[c] = np.log1p(num[c].clip(lower=0))
        num = (num - num.mean()) / (num.std() + 1e-8)

        cat_cols = ["workclass","education","marital-status","occupation",
                    "relationship","native-country"]
        for c in cat_cols:
            df[c] = df[c].fillna("Unknown").astype(str)
        onehot = pd.get_dummies(df[cat_cols], columns=cat_cols).astype(np.float32)

        feats = np.hstack([num.values.astype(np.float32), onehot.values]).astype(np.float32)
        self._finalize(feats, df["label"], df["race_code"], df["gender_code"], "adult")

    def as_torch_dataset(self):
        return TensorDataset4(self.features, self.labels, self.genders, self.races)

In [4]:
# ══════════════════════════════════════════════════════════════════════
# LOAD → SPLIT → CANARIES (stratified by race, disjoint from train/val)
# Wrapped in construct_split() so the auto-tuner can re-invoke when k changes.
# ══════════════════════════════════════════════════════════════════════

def construct_split(k_override=None):
    """Build full_ds, train/val split, and stratified canary pool.

    Sets the following module-level globals (used by Cells 5, 9, 13, 15, 18):
      full_ds, N, race_all, input_dim, get_subgroups, get_targets,
      train_tf, eval_tf,
      canary_in_pool, canary_out_pool, canary_set,
      train_idx, val_idx, remaining,
      membership, canary_img_indices, canary_labels, canary_subgroups, k
    """
    global full_ds, N, race_all, input_dim, get_subgroups, get_targets
    global train_tf, eval_tf
    global canary_in_pool, canary_out_pool, canary_set
    global train_idx, val_idx, remaining
    global membership, canary_img_indices, canary_labels, canary_subgroups, k
    global POS_RATE, TRIVIAL_UTILITY

    if k_override is not None:
        CFG["k"] = int(k_override)

    k = CFG["k"]

    audit_rng = np.random.default_rng(AUDIT_SEED)
    all_indices = np.arange(N) if 'N' in globals() else None

    # ── Load dataset (only once — full_ds doesn't depend on k) ──
    if 'full_ds' not in globals() or full_ds is None:

        if DATASET == "utkface":
            _train_tf = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(0.5),
                transforms.ColorJitter(0.2, 0.2, 0.1),
                transforms.ToTensor(),
                transforms.Normalize(
                    [0.485, 0.456, 0.406],
                    [0.229, 0.224, 0.225]
                )
            ])

            _eval_tf = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    [0.485, 0.456, 0.406],
                    [0.229, 0.224, 0.225]
                )
            ])

            _full_ds = UTKFaceDataset(
                DATA_ROOT / "UTKFace",
                transform=None
            )

            _N = len(_full_ds)

            _get_subgroups = lambda idx: {
                "race": np.array(
                    [_full_ds.samples[i]["race"] for i in idx],
                    dtype=np.int64
                ),
                "gender": np.array(
                    [_full_ds.samples[i]["gender"] for i in idx],
                    dtype=np.int64
                ),
            }

            _get_targets = lambda idx: np.array(
                [_full_ds.samples[i]["age"] for i in idx],
                dtype=np.float32
            )

            _input_dim = None

            _race_all = np.array(
                [s["race"] for s in _full_ds.samples],
                dtype=np.int64
            )

        else:
            loader = TabularLoader(DATASET, DATA_ROOT)

            _full_ds = loader.as_torch_dataset()
            _N = len(_full_ds)

            _get_subgroups = lambda idx: {
                "race": loader.races[np.asarray(idx)],
                "gender": loader.genders[np.asarray(idx)],
            }

            _get_targets = lambda idx: loader.labels[np.asarray(idx)]

            _input_dim = loader.input_dim
            _race_all = loader.races

            _train_tf = None
            _eval_tf = None

        # Persist to globals ONCE — these don't change with k
        for n, v in [
            ("full_ds", _full_ds),
            ("N", _N),
            ("get_subgroups", _get_subgroups),
            ("get_targets", _get_targets),
            ("input_dim", _input_dim),
            ("race_all", _race_all),
            ("train_tf", _train_tf),
            ("eval_tf", _eval_tf),
        ]:
            globals()[n] = v

    # ── Classification reference values ──
    POS_RATE = (
        float(np.asarray(get_targets(range(N))).mean())
        if CFG["task"] == "classification"
        else 0.5
    )

    TRIVIAL_UTILITY = (
        max(POS_RATE, 1.0 - POS_RATE)
        if CFG["task"] == "classification"
        else None
    )

    # ── Stratified Canary Selection ──
    canary_in_pool = []
    canary_out_pool = []

    per_group = max(k // 5, 60)

    for rv in range(5):
        pool = np.where(race_all == rv)[0]

        if len(pool) == 0:
            continue

        audit_rng.shuffle(pool)

        take = min(per_group * 2, len(pool))

        canary_in_pool.extend(
            pool[:take // 2].tolist()
        )

        canary_out_pool.extend(
            pool[take // 2:take].tolist()
        )

    canary_set = (
        set(canary_in_pool) |
        set(canary_out_pool)
    )

    # ── Fill remaining canaries if stratified pools are smaller than k ──
    available = np.array([
        i for i in np.arange(N)
        if i not in canary_set
    ])

    audit_rng.shuffle(available)

    ptr = 0

    while len(canary_in_pool) < k:
        canary_in_pool.append(
            int(available[ptr])
        )
        ptr += 1

    while len(canary_out_pool) < k:
        canary_out_pool.append(
            int(available[ptr])
        )
        ptr += 1

    canary_in_pool = np.array(
        canary_in_pool[:k]
    )

    canary_out_pool = np.array(
        canary_out_pool[:k]
    )

    canary_set = (
        set(canary_in_pool.tolist()) |
        set(canary_out_pool.tolist())
    )

    # ── Train/Val Split (strictly disjoint from canaries) ──
    remaining = np.array([
        i for i in np.arange(N)
        if i not in canary_set
    ])

    n_val = int(
        len(remaining) * CFG["val_frac"]
    )

    train_idx = (
        remaining[:-n_val].tolist()
        if n_val > 0
        else remaining.tolist()
    )

    val_idx = (
        remaining[-n_val:].tolist()
        if n_val > 0
        else []
    )

    # ── Canary Construction ──
    membership = torch.tensor(
        audit_rng.integers(
            0,
            2,
            size=k
        ).astype(bool),
        dtype=torch.bool
    )

    canary_img_indices = [
        int(canary_in_pool[i])
        if m
        else int(canary_out_pool[i])
        for i, m in enumerate(membership.tolist())
    ]

    # Canary labels sampled from validation targets
    val_targets = get_targets(val_idx)

    audit_rng.shuffle(val_targets)

    canary_labels = torch.as_tensor(
        val_targets[:k],
        dtype=torch.float32
    )

    canary_subgroups = get_subgroups(
        canary_img_indices
    )

    # ── Diagnostics ──
    print(
        f"[construct_split k={k}] "
        f"N={N} | "
        f"train={len(train_idx)} "
        f"val={len(val_idx)} | "
        f"k={k} "
        f"(IN={int(membership.sum())}, "
        f"OUT={int((~membership).sum())})"
    )

    if TRIVIAL_UTILITY is not None:
        print(
            f"POS_RATE={POS_RATE:.3f}  "
            f"TRIVIAL_UTILITY={TRIVIAL_UTILITY:.3f}"
        )
    else:
        print(
            f"regression task, "
            f"POS_RATE={POS_RATE}"
        )

    # ── Assertions ──
    assert (
        0.35 * k < membership.sum() < 0.65 * k
    ), "Membership too lopsided — reseed AUDIT_SEED"

    assert set(canary_img_indices).isdisjoint(
        set(train_idx)
    ), "CANARY LEAK INTO TRAIN!"

    assert set(canary_img_indices).isdisjoint(
        set(val_idx)
    ), "CANARY LEAK INTO VAL!"

    return dict(
        k=k,
        N=N,
        n_train=len(train_idx),
        n_val=len(val_idx),
        n_in=int(membership.sum()),
        n_out=int((~membership).sum())
    )


# Initial construction (k from CFG)
construct_split()

print("✓ Canary/train/val disjointness asserted")

[adult] N=32561 features=101 pos_rate=0.241
    race 0: n= 27816 pos_rate=0.256
    race 1: n=  3124 pos_rate=0.124
    race 2: n=  1039 pos_rate=0.266
    race 3: n=   311 pos_rate=0.116
    race 4: n=   271 pos_rate=0.092
[construct_split k=500] N=32561 | train=26827 val=4734 | k=500 (IN=235, OUT=265)
POS_RATE=0.241  TRIVIAL_UTILITY=0.759
✓ Canary/train/val disjointness asserted


In [5]:
# ══════════════════════════════════════════════════════════════════════
# BUILD PHASE DATASETS — defines train_ds_full, val_loader, canary_ds,
# in_canary_ds, augmented_train_ds. Wrapped in build_phase_datasets() so the
# auto-tuner can rebuild when k changes.
# ══════════════════════════════════════════════════════════════════════

class TransformSubset(Dataset):
    """Subset that applies a transform to element 0 and returns 5-tuple
    (x, y, gender, race, local_index)."""
    def __init__(self, base, indices, transform=None):
        self.base, self.indices, self.tf = base, list(indices), transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        item = self.base[self.indices[i]]
        x = self.tf(item[0]) if (self.tf is not None and not torch.is_tensor(item[0])) else item[0]
        return x, item[1], item[2], item[3], i


class CanaryDataset:
    """Holds canary tensors on device; provides IN-subset for training augmentation."""
    def __init__(self, full_ds, img_indices, labels, transform):
        self.labels = labels.to(device)
        feats = []
        for gi in img_indices:
            item = full_ds[gi]
            x = transform(item[0]) if (transform is not None and not torch.is_tensor(item[0])) else item[0]
            feats.append(x)
        self.features = torch.stack(feats).to(device)
    def get_in_dataset(self):
        in_pos = [i for i, m in enumerate(membership.tolist()) if m]
        return _InCanarySubset(self.features[in_pos], self.labels[in_pos]), in_pos
    def get_all(self):
        return self.features, self.labels


class _InCanarySubset(Dataset):
    """IN canaries; returns local_index = -1 to mark them as canaries."""
    def __init__(self, feats, labs):
        self.f, self.l = feats.cpu(), labs.cpu()
        n = len(labs)
        self.g = torch.full((n,), -1, dtype=torch.long)
        self.r = torch.full((n,), -1, dtype=torch.long)
    def __len__(self): return len(self.l)
    def __getitem__(self, i):
        return self.f[i], self.l[i], self.g[i], self.r[i], -1


def build_phase_datasets():
    """Rebuild all phase datasets from current Cell 4 globals.
    Sets module-level: train_ds_full, val_ds, val_loader, canary_ds,
    in_canary_ds, in_canary_pos, augmented_train_ds.
    Call this whenever the canary split changes (e.g., k was overridden)."""
    global train_ds_full, val_ds, val_loader
    global canary_ds, in_canary_ds, in_canary_pos, augmented_train_ds

    train_ds_full = TransformSubset(full_ds, train_idx, transform=train_tf)
    val_ds        = TransformSubset(full_ds, val_idx,   transform=eval_tf)
    val_loader    = DataLoader(val_ds, batch_size=CFG["batch_size"],
                               shuffle=False, num_workers=0, pin_memory=True)

    canary_ds      = CanaryDataset(full_ds, canary_img_indices, canary_labels, transform=eval_tf)
    in_canary_ds, in_canary_pos = canary_ds.get_in_dataset()
    augmented_train_ds = ConcatDataset([train_ds_full, in_canary_ds])

    print(f"[build_phase_datasets] train_ds_full={len(train_ds_full)} "
          f"val_ds={len(val_ds)} canaries={len(canary_ds.labels)} "
          f"IN={len(in_canary_ds)} augmented={len(augmented_train_ds)}")


# Initial build
build_phase_datasets()


[build_phase_datasets] train_ds_full=26827 val_ds=4734 canaries=500 IN=235 augmented=27062


In [6]:
class GNBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(min(32, planes), planes)
        self.relu  = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(min(32, planes), planes)
        self.downsample = downsample
    def forward(self, x):
        out = self.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        if self.downsample: x = self.downsample(x)
        return self.relu(out + x)

class ResNet18GN(nn.Module):
    def __init__(self):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.gn1    = nn.GroupNorm(min(32, 64), 64)
        self.relu   = nn.ReLU(inplace=False)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64,  2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)
    def _make_layer(self, planes, blocks, stride):
        ds = (nn.Sequential(nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                            nn.GroupNorm(min(32, planes), planes))
              if stride != 1 or self.inplanes != planes else None)
        layers = [GNBlock(self.inplanes, planes, stride, ds)]
        self.inplanes = planes
        for _ in range(1, blocks): layers.append(GNBlock(planes, planes))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.maxpool(self.relu(self.gn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1)).squeeze(1)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden=(256, 128), num_outputs=1):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, num_outputs))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

# ═══════════════════════════════════════════════════════════════
# NEW: Deep Tabular ResNet (Decision D-08)
# Uses LayerNorm (per-sample safe for Opacus/DP-SGD)
# ═══════════════════════════════════════════════════════════════
class TabularResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.net(x))

class DeepTabularResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=6, num_outputs=1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        self.blocks = nn.Sequential(*[TabularResBlock(hidden_dim) for _ in range(num_blocks)])
        self.output_head = nn.Linear(hidden_dim, num_outputs)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.blocks(x)
        return self.output_head(x).squeeze(-1)

def build_logreg(input_dim):
    return nn.Linear(input_dim, 1)
    
def build_model():
    if CFG["model"] == "resnet18":
        return ResNet18GN().to(device)
    if CFG["model"] == "deep_resnet":
        # 6 blocks of 256 hidden units -> high capacity for memorization
        return DeepTabularResNet(input_dim, hidden_dim=256, num_blocks=6).to(device)
    if CFG["model"] == "mlp":
        return TabularMLP(input_dim).to(device)
    return build_logreg(input_dim).to(device)

def loss_fn(preds, targets):
    p = preds.reshape(-1)
    t = targets.reshape(-1)
    if CFG["task"] == "regression":
        return (p - t).abs()
    return F.binary_cross_entropy_with_logits(p, t, reduction="none")

In [7]:
class ChiSqDRO:
    """Hashimoto χ²-DRO with dimensionless dual ascent."""
    def __init__(self, n_train, alpha, eta, lambda_init, warmup_epochs,
                 w_clip=(0.2, 5.0), max_ratio=10.0, uniform_mix=0.5):
        self.n_train = n_train
        self.q = np.ones(n_train) / n_train
        self.alpha = float(alpha)
        self.eta = float(eta)
        self.lambda_max = 1.0 / (2.0 * self.alpha) if self.alpha > 0 else 1.0
        # Convert lambda_init to normalized nu
        self.nu = float(np.clip(lambda_init / self.lambda_max, 0.0, 1.0)) if self.lambda_max > 0 else 0.0
        self.warmup = warmup_epochs
        self.w_clip, self.max_ratio, self.uniform_mix = w_clip, max_ratio, uniform_mix
        self.history = {"mu": [], "V": [], "lambda": [], "nu": []}

    @property
    def lambda_t(self):
        return self.nu * self.lambda_max

    @property
    def activation_ratio(self):
        return self.nu

    def update(self, losses, epoch):
        mu_t = float(np.sum(self.q * losses))
        V_t = float(np.sum(self.q * (losses - mu_t) ** 2))
        # NORMALIZED dual ascent — dimensionless
        excess = (V_t / self.alpha - 1.0) if self.alpha > 0 else 0.0
        self.nu = float(np.clip(self.nu + self.eta * excess, 0.0, 1.0))
        lam = self.lambda_t
        w = np.clip(1.0 + lam * (losses - mu_t), self.w_clip[0], self.w_clip[1])
        q_new = np.maximum(self.q * w, 1e-12); q_new /= q_new.sum()
        u = 1.0 / self.n_train
        q_new = np.clip(q_new, u / self.max_ratio, u * self.max_ratio); q_new /= q_new.sum()
        self.q = q_new
        self.history["mu"].append(mu_t); self.history["V"].append(V_t)
        self.history["lambda"].append(lam); self.history["nu"].append(self.nu)
        return mu_t, V_t

    def sample_subepoch(self, frac):
        u = 1.0 / self.n_train
        p = (1.0 - self.uniform_mix) * self.q + self.uniform_mix * u
        p /= p.sum()
        n = int(self.n_train * frac)
        idx = np.random.choice(self.n_train, size=n, p=p, replace=True)
        return np.array(list(dict.fromkeys(idx.tolist())))

    def get_sample_weights(self, n):
        w = self.q / (1.0 / self.n_train)
        w = np.clip(w, self.w_clip[0], self.w_clip[1])
        return torch.as_tensor(w / w.mean(), dtype=torch.float32)

    def state(self):
        return {"q": self.q.copy(), "nu": self.nu, "lambda_t": self.lambda_t, "history": self.history}

    def load_state(self, st):
        self.q = np.asarray(st["q"])
        self.nu = float(st.get("nu", 0.0))
        if self.nu == 0.0 and "lambda_t" in st and self.lambda_max > 0:
            self.nu = float(np.clip(st["lambda_t"] / self.lambda_max, 0.0, 1.0))
        self.history = st.get("history", {"mu": [], "V": [], "lambda": [], "nu": []})

In [8]:
def save_checkpoint(ckpt_dir: Path, epoch, model, optimizer, history, dro_state, extra):
    """Atomic write: tmp file → os.replace. Safe against crashes & parallel runs."""
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    base = model._module if hasattr(model, "_module") else model
    payload = {
        "epoch": epoch,
        "model_state_dict":  {k: v.cpu() for k, v in base.state_dict().items()},
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "dro_state": dro_state,
        "extra": extra,
        "timestamp": datetime.now().isoformat(),
    }
    tmp = ckpt_dir / "latest.pt.tmp"
    torch.save(payload, tmp)
    os.replace(tmp, ckpt_dir / "latest.pt")
    torch.save(payload, ckpt_dir / f"epoch_{epoch:03d}.pt")

def load_checkpoint(ckpt_dir: Path):
    p = ckpt_dir / "latest.pt"
    return torch.load(p, map_location="cpu") if p.exists() else None

def save_json_atomic(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o:
                  o.tolist() if isinstance(o, np.ndarray)
                  else float(o) if isinstance(o, (np.floating,)) else
                  int(o) if isinstance(o, (np.integer,)) else
                  bool(o) if isinstance(o, np.bool_) else str(o))
    os.replace(tmp, path)


In [9]:
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager
import copy, math

def _uniform_loader(ds, batch_size, workers=0):
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=workers, pin_memory=True)


def evaluate_fairness(model, loader):
    base = model._module if hasattr(model, "_module") else model
    base.eval()
    n = 0; err_sum = 0.0; perr_sum = 0.0
    g_err, g_perr, g_n = {0:0.0,1:0.0}, {0:0.0,1:0.0}, {0:0,1:0}
    r_err, r_perr, r_n = {}, {}, {}
    stds, means = [], []
    MIN_N = 30

    with torch.no_grad():
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            g, r = batch[2], batch[3]
            raw = base(x).reshape(-1)
            y = y.reshape(-1)

            if CFG["task"] == "regression":
                pc = raw.cpu()
                err = (raw - y).abs().cpu()
                perr = err
            else:
                p = torch.sigmoid(raw)
                pc = p.cpu()
                err = (p.round() != y).float().cpu()
                perr = (p - y).abs().cpu()

            stds.append(float(pc.std()))
            means.append(float(pc.mean()))

            valid = (g >= 0)
            nv = int(valid.sum())
            n += nv
            err_sum += err[valid].sum().item()
            perr_sum += perr[valid].sum().item()

            for gv in (0, 1):
                m = (g == gv) & valid
                g_err[gv] += err[m].sum().item()
                g_perr[gv] += perr[m].sum().item()
                g_n[gv] += int(m.sum())

            for rv in torch.unique(r[valid]):
                rv = int(rv.item())
                m = (r == rv) & valid
                r_err[rv] = r_err.get(rv, 0.0) + err[m].sum().item()
                r_perr[rv] = r_perr.get(rv, 0.0) + perr[m].sum().item()
                r_n[rv] = r_n.get(rv, 0) + int(m.sum())

    male, female = g_err[0]/max(g_n[0],1), g_err[1]/max(g_n[1],1)
    male_p, female_p = g_perr[0]/max(g_n[0],1), g_perr[1]/max(g_n[1],1)
    re_ = [r_err[k]/max(r_n[k],1) for k in r_err if r_n[k] > MIN_N]
    rp_ = [r_perr[k]/max(r_n[k],1) for k in r_perr if r_n[k] > MIN_N]

    return {
        "utility": err_sum/max(n,1),
        "utility_prob": perr_sum/max(n,1),
        "gender_gap": abs(male-female),
        "gender_gap_prob": abs(male_p-female_p),
        "race_gap": (max(re_)-min(re_)) if len(re_) > 1 else 0.0,
        "race_gap_prob": (max(rp_)-min(rp_)) if len(rp_) > 1 else 0.0,
        "male": male,
        "female": female,
        "pred_std": float(np.mean(stds)) if stds else 0.0,
        "pred_mean": float(np.mean(means)) if means else 0.0,
        "race_err": {k: r_err[k]/max(r_n[k],1) for k in r_err},
        "race_n": dict(r_n),
    }


TRIVIAL_LOSS = math.log(2.0)


def measure_loss_variance(model, n_max=20000, seed=0):
    """Per-sample loss variance on a training subsample (cheap, epoch-wise)."""
    base = model._module if hasattr(model, "_module") else model
    base.eval()
    n = len(train_ds_full)
    idx = np.random.default_rng(seed).choice(n, size=min(n_max, n), replace=False)
    losses = []

    with torch.no_grad():
        for b in DataLoader(
            Subset(train_ds_full, idx.tolist()),
            batch_size=512,
            shuffle=False
        ):
            losses.append(
                loss_fn(base(b[0].to(device)), b[1].to(device)).cpu().numpy()
            )

    return float(np.var(np.concatenate(losses)))


def train_one_run(is_dro, sigma, dro, epochs, ckpt_dir,
                  resume_epoch=0, initial_params=None, initial_losses=None):

    model = build_model()

    if initial_params is None:
        initial_params = {
            n: p.detach().clone().cpu()
            for n, p in model.named_parameters()
        }

    optimizer = optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"]
    )

    use_dp = sigma is not None
    privacy_engine = None

    if use_dp:
        privacy_engine = PrivacyEngine()
        base_loader = _uniform_loader(
            augmented_train_ds,
            CFG["batch_size"]
        )

        model, optimizer, base_loader = privacy_engine.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=base_loader,
            noise_multiplier=sigma,
            max_grad_norm=DP_MAX_GRAD_NORM
        )
    else:
        base_loader = _uniform_loader(
            augmented_train_ds,
            CFG["batch_size"]
        )

    base = model._module if hasattr(model, "_module") else model

    c_feats, c_labs = canary_ds.get_all()

    if initial_losses is None:
        with torch.no_grad():
            initial_losses = loss_fn(
                base(c_feats),
                c_labs
            ).cpu().numpy()

    history = {
        "train_loss": [],
        "train_loss_unweighted": [],
        "V_train": [],
        "utility": [],
        "gender_gap": [],
        "race_gap": [],
        "epsilon": [],
        "lambda_t": [],
        "V_t": [],
        "activation_ratio": [],
        "pred_std": [],
        "early_stopped": False,
        "trivial_collapse": False,
        "stopped_epoch": epochs
    }

    t0 = time.time()
    tag = "DRO" if is_dro else "ERM"
    best_util, best_state, bad_epochs, trivial_count = float("inf"), None, 0, 0
    patience = 8

    try:
        for epoch in range(resume_epoch + 1, epochs + 1):

            dro_weights = None

            if is_dro and epoch > dro.warmup:
                dro_weights = dro.get_sample_weights(dro.n_train)

            if is_dro and epoch > dro.warmup and not use_dp:
                sub_idx = dro.sample_subepoch(CFG["sub_epoch_frac"])
                loader = _uniform_loader(
                    ConcatDataset([
                        Subset(train_ds_full, sub_idx),
                        in_canary_ds
                    ]),
                    CFG["batch_size"]
                )
            else:
                loader = base_loader

            model.train()
            ep_loss, ep_loss_unw, ep_n = 0.0, 0.0, 0

            def _step(batch):
                nonlocal ep_loss, ep_loss_unw, ep_n

                x, y, idx = (
                    batch[0].to(device),
                    batch[1].to(device),
                    batch[4]
                )

                optimizer.zero_grad()

                per_sample = loss_fn(model(x), y)

                if dro_weights is not None:
                    w = torch.ones_like(per_sample)
                    tm = idx >= 0

                    if tm.any():
                        w[tm] = dro_weights[idx[tm]].to(device)

                    loss = (w * per_sample).mean()
                else:
                    loss = per_sample.mean()

                loss.backward()
                optimizer.step()

                bs = len(y)

                ep_loss += loss.item() * bs
                ep_loss_unw += per_sample.mean().item() * bs
                ep_n += bs

            if use_dp:
                with BatchMemoryManager(
                    data_loader=loader,
                    max_physical_batch_size=CFG["phys_batch"],
                    optimizer=optimizer
                ) as mem_loader:
                    for batch in mem_loader:
                        _step(batch)
            else:
                for batch in loader:
                    _step(batch)

            avg_loss = ep_loss / max(ep_n, 1)
            avg_loss_unw = ep_loss_unw / max(ep_n, 1)

            # ── End-of-epoch DRO dual update over the FULL training set ──
            V_t = 0.0

            if is_dro and epoch > dro.warmup:
                base.eval()
                all_losses = []

                with torch.no_grad():
                    for b in DataLoader(
                        train_ds_full,
                        batch_size=512,
                        shuffle=False
                    ):
                        all_losses.append(
                            loss_fn(
                                base(b[0].to(device)),
                                b[1].to(device)
                            ).cpu().numpy()
                        )

                mu_t, V_t = dro.update(
                    np.concatenate(all_losses),
                    epoch
                )

            # Variance probe every epoch → this is what α calibration consumes
            V_epoch = (
                V_t
                if V_t > 0
                else measure_loss_variance(model, seed=epoch)
            )

            fair = evaluate_fairness(model, val_loader)
            eps = (
                privacy_engine.get_epsilon(DP_DELTA)
                if privacy_engine
                else 0.0
            )

            history["train_loss"].append(avg_loss)
            history["train_loss_unweighted"].append(avg_loss_unw)
            history["V_train"].append(V_epoch)
            history["utility"].append(fair["utility"])
            history["gender_gap"].append(fair["gender_gap"])
            history["race_gap"].append(fair["race_gap"])
            history["epsilon"].append(eps)
            history["lambda_t"].append(
                dro.lambda_t if is_dro else 0.0
            )
            history["V_t"].append(V_t)
            history["activation_ratio"].append(
                dro.activation_ratio if is_dro else 0.0
            )
            history["pred_std"].append(
                fair.get("pred_std", 1.0)
            )

            # ── Guards ──
            if fair["utility"] < best_util - 1e-4:
                best_util = fair["utility"]
                best_state = copy.deepcopy(base.state_dict())
                bad_epochs = 0
            else:
                bad_epochs += 1

            # Collapse test: constant output, OR utility no better than the trivial
            # predictor. Uses UNWEIGHTED loss so DRO reweighting can't fake a trigger.
            collapsed = False

            if CFG["task"] == "classification":
                collapsed = (
                    fair.get("pred_std", 1.0) < 0.02
                ) or (
                    fair["utility"] >= TRIVIAL_UTILITY - 0.005
                )

            trivial_count = (
                trivial_count + 1
                if collapsed
                else 0
            )

            if (
                trivial_count >= 3
                and epoch >= CFG.get("min_epochs", 20)
            ):
                history["trivial_collapse"] = True
                history["early_stopped"] = True
                history["stopped_epoch"] = epoch

                print(
                    f"  ⚠️ [{tag}] TRIVIAL COLLAPSE at ep {epoch} "
                    f"(pred_std={fair.get('pred_std',0):.4f}, "
                    f"U={fair['utility']:.3f} "
                    f"vs trivial {TRIVIAL_UTILITY:.3f}). "
                    f"Restoring best weights."
                )
                break

            if (
                bad_epochs >= CFG.get("patience", 12)
                and epoch >= CFG.get("min_epochs", 20)
            ):
                history["early_stopped"] = True
                history["stopped_epoch"] = epoch

                print(
                    f"  ⚠️ [{tag}] Early stop at epoch {epoch}."
                )
                break

            if epoch % 5 == 0 or epoch == epochs:
                print(
                    f"  [{tag} σ={sigma}] Ep {epoch:2d} | "
                    f"L={avg_loss_unw:.3f} | "
                    f"U={fair['utility']:.3f} | "
                    f"ε={eps:.2f} | "
                    f"ν={history['activation_ratio'][-1]*100:.0f}% | "
                    f"V={V_epoch:.4g} | "
                    f"{time.time()-t0:.0f}s"
                )

                save_checkpoint(
                    ckpt_dir,
                    epoch,
                    model,
                    optimizer,
                    history,
                    dro.state() if is_dro else None,
                    {
                        "initial_losses": initial_losses,
                        "initial_params": {
                            n: p.cpu()
                            for n, p in initial_params.items()
                        }
                    }
                )

    except KeyboardInterrupt:
        print(f"  ⚠️ INTERRUPTED at epoch {epoch}")

        save_checkpoint(
            ckpt_dir,
            epoch,
            model,
            optimizer,
            history,
            dro.state() if is_dro else None,
            {
                "initial_losses": initial_losses,
                "initial_params": {
                    n: p.cpu()
                    for n, p in initial_params.items()
                }
            }
        )

        raise

    if best_state is not None:
        base.load_state_dict(best_state)

    base.eval()

    with torch.no_grad():
        final_losses = loss_fn(
            base(c_feats),
            c_labs
        ).cpu().numpy()

    return {
        "model": model,
        "history": history,
        "initial_params": initial_params,
        "bb_scores": initial_losses - final_losses,
        "epsilon_spent": history["epsilon"][-1],
        "final_metrics": fair
    }

In [10]:
def compute_attack_scores(run_result, batch_size=64):
    theta0 = run_result["initial_params"]
    base_model = (run_result["model"]._module
                  if hasattr(run_result["model"], "_module") else run_result["model"])

    wb_model = build_model()
    wb_model.load_state_dict(base_model.state_dict())
    wb_model.to(device); wb_model.eval()

    thetaT  = {n: p.detach().cpu() for n, p in wb_model.named_parameters()}
    delta_w = torch.cat([(thetaT[n] - theta0[n].cpu()).view(-1) for n in thetaT])

    c_feats, c_labs = canary_ds.get_all()
    k = len(c_labs)
    wb_scores      = np.zeros(k, dtype=np.float64)
    preclip_norms  = np.zeros(k, dtype=np.float64)

    for s in range(0, k, batch_size):
        e = min(s + batch_size, k)
        for i in range(s, e):
            wb_model.zero_grad(set_to_none=True)
            xi = c_feats[i:i+1].clone().detach()
            yi = c_labs[i:i+1]
            li = loss_fn(wb_model(xi), yi).sum()
            grads = torch.autograd.grad(li, list(wb_model.parameters()))
            g_flat = torch.cat([g.view(-1) for g in grads]).cpu()

            norm_val = g_flat.norm().item()
            preclip_norms[i] = norm_val
            clip_f = min(1.0, DP_MAX_GRAD_NORM / (norm_val + 1e-8))
            wb_scores[i] = torch.dot(delta_w, g_flat * clip_f).item()

    del wb_model; torch.cuda.empty_cache()
    return run_result["bb_scores"], wb_scores, preclip_norms


In [11]:
def steinke_audit(scores: np.ndarray, mem: np.ndarray, gamma=AUDIT_GAMMA):
    m = np.asarray(mem).astype(int); k = len(scores)
    thresholds = np.linspace(scores.min() - 1e-9, scores.max() + 1e-9, 1000)
    best_V, best_tau = 0, thresholds[0]
    for tau in thresholds:
        V = int((((scores > tau).astype(int)) == m).sum())
        if V > best_V: best_V, best_tau = V, tau

    p_null = float(stats.binom.sf(best_V - 1, k, 0.5))
    def p_val(eps):
        return stats.binom.sf(best_V - 1, k, math.exp(eps) / (1 + math.exp(eps)))
    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if p_val(mid) > gamma: lo = mid
        else: hi = mid
    eps_lb = lo if p_val(lo) <= gamma else 0.0

    in_s, out_s = scores[m == 1], scores[m == 0]
    pooled = math.sqrt((in_s.var() + out_s.var()) / 2) + 1e-12
    try: auc = float(roc_auc_score(m, scores))
    except Exception: auc = 0.5
    return {"eps_lb": float(eps_lb), "V": best_V, "k": k, "threshold": float(best_tau),
            "p_value": p_null, "attack_acc": best_V / k, "auc": auc,
            "delta": float(in_s.mean() - out_s.mean()),
            "cohens_d": float((in_s.mean() - out_s.mean()) / pooled)}


def paired_tests(scores_a: np.ndarray, scores_b: np.ndarray):
    d = np.asarray(scores_b) - np.asarray(scores_a)
    if np.allclose(d, 0):
        return {"wilcoxon_p": 1.0, "cohens_d_paired": 0.0, "mean_diff": 0.0}
    try:
        w_stat, w_p = stats.wilcoxon(d, alternative="two-sided")
    except ValueError:
        w_stat, w_p = np.nan, 1.0
    return {"wilcoxon_p": float(w_p),
            "cohens_d_paired": float(d.mean() / (d.std() + 1e-12)),
            "mean_diff": float(d.mean())}


In [12]:
def subgroup_analysis(wb_scores, preclip_norms, mem, subgroups, min_n=10):
    m = np.asarray(mem).astype(int)
    out = {}
    for axis in ("race", "gender"):
        labels = np.asarray(subgroups[axis])
        out[axis] = {}
        for gv in np.unique(labels):
            if gv < 0: continue
            mask = labels == gv
            n_in  = int((mask & (m == 1)).sum())
            n_out = int((mask & (m == 0)).sum())
            if n_in < min_n or n_out < min_n: continue
            try:
                auc = float(roc_auc_score(m[mask], wb_scores[mask]))
            except Exception:
                auc = 0.5
            name = (RACE_MAP if axis == "race" else GENDER_MAP).get(int(gv), f"{axis}_{gv}")
            out[axis][name] = {
                "auc": auc, "n_in": n_in, "n_out": n_out,
                "preclip_grad_norm":     float(preclip_norms[mask].mean()),
                "preclip_grad_norm_std": float(preclip_norms[mask].std()),
            }
        aucs  = [v["auc"]                for v in out[axis].values()]
        norms = [v["preclip_grad_norm"]  for v in out[axis].values()]
        out[axis]["_disparity"]      = (max(aucs)  - min(aucs))  if len(aucs)  > 1 else 0.0
        out[axis]["_norm_disparity"] = (max(norms) - min(norms)) if len(norms) > 1 else 0.0
    return out


In [13]:
CALIB_PATH = EXP_ROOT / "alpha_calibration.json"

def calibrate_alpha_from_erm(erm_result, frac=None):
    """α = frac × CONVERGED ERM train-loss variance.

    Critical: variance decays as the model learns. Calibrating from a 3-epoch
    pilot sets α above the V_t the DRO run will actually see, so (V_t − α) < 0
    and λ is pinned at 0 for the entire run. Using the converged ERM variance
    keeps α just below V_t so the dual variable can rise.
    """
    frac = frac if frac is not None else CFG.get("dro_alpha_frac", 0.7)
    Vs = [v for v in erm_result["history"].get("V_train", []) if v > 0]
    if len(Vs) < 3:
        raise RuntimeError("ERM run recorded too few V_train points to calibrate α")
    V_conv = float(np.mean(Vs[-3:]))
    return frac * V_conv, V_conv
    
def calibrate_alpha(force=False):
    if CALIB_PATH.exists() and not force:
        cal = json.load(open(CALIB_PATH))
        print(f"📂 α calibration loaded: α={cal['alpha']:.6g} (V_nat={cal['V_natural']:.4g})")
        return cal["alpha"], cal["V_natural"]

    print("🔬 Running α-calibration pilot (3 epochs, no DP, no DRO)...")
    model = build_model()
    opt   = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    loader = _uniform_loader(augmented_train_ds, CFG["batch_size"])
    for ep in range(3):
        model.train()
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            opt.zero_grad(); loss_fn(model(x), y).mean().backward(); opt.step()

    model.eval()
    losses = []
    with torch.no_grad():
        for batch in DataLoader(train_ds_full, batch_size=256, shuffle=False):
            x, y = batch[0].to(device), batch[1].to(device)
            losses.append(loss_fn(model(x), y).cpu().numpy())
    losses = np.concatenate(losses)
    V_nat  = float(np.var(losses))
    alpha  = 0.8 * V_nat
    save_json_atomic(CALIB_PATH, {"V_natural": V_nat, "alpha": alpha,
                                  "mean_loss": float(losses.mean()),
                                  "timestamp": datetime.now().isoformat()})
    print(f"✅ V_natural={V_nat:.4g} → α={alpha:.6g} (0.8·V_nat)")
    del model; gc.collect(); torch.cuda.empty_cache()
    return alpha, V_nat


In [14]:
def validate_dro_result(res, alpha_used):
    hist = res["history"]
    nu_traj = hist.get("activation_ratio", [])
    nu_max   = max(nu_traj) if nu_traj else 0.0      # ← KEY CHANGE: max, not final
    utils    = hist["utility"]

    # A) trivial collapse → optimisation failure
    if hist.get("trivial_collapse"):
        return False, "trivial_collapse", None

    # B) utility diverged
    if len(utils) > 2 and utils[-1] > 1.6 * min(utils):
        if nu_max > 0.95:
            return False, "lambda_saturation_divergence", alpha_used * 2.0
        return False, "utility_diverging", None       # lr problem, not α

    # C) NEVER activated at any point → α too high
    if nu_max < 0.05:
        return False, "lambda_dormant", alpha_used * 0.5

    # D) Activated during the run (even if it converged back to 0) = HEALTHY
    return True, "ok", None

In [15]:
def run_cell(eps_label, sigma, is_dro, alpha, attempt=1, tried_alphas=None):
    if tried_alphas is None: tried_alphas = []

    tag = "dro" if is_dro else "erm"
    cell_dir     = EXP_ROOT / eps_label / tag
    results_path = cell_dir / "result.json"

    # Resume / skip logic
    if results_path.exists():
        prev = json.load(open(results_path))
        prev_overrides = prev.get("overrides_applied", {})
        prev_alpha = prev.get("alpha_used", None)

        # Invalidate if overrides changed OR α changed
        if prev_overrides != _ACTIVE_OVERRIDES:
            print(f"♻️  {eps_label}/{tag}: overrides changed → clearing & re-running")
            for f in cell_dir.glob("*.pt"): f.unlink()
            results_path.unlink()
        elif is_dro and prev_alpha is not None and abs(prev_alpha - alpha) / max(alpha, 1e-9) > 0.01:
            print(f"♻️  {eps_label}/{tag}: α changed ({prev_alpha:.6g} → {alpha:.6g}) → re-running")
            for f in cell_dir.glob("*.pt"): f.unlink()
            results_path.unlink()
        elif not is_dro or validate_dro_result(prev, prev.get("alpha_used", alpha))[0]:
            if prev.get("validated", False) or not is_dro:
                print(f"⏭  {eps_label}/{tag}: complete & valid — skipping")
                return prev
            print(f"♻️  {eps_label}/{tag}: invalid ({prev.get('invalid_reason','?')}) — re-running")

    cell_dir.mkdir(parents=True, exist_ok=True)

    # Pick up ChiSqDRO constructor overrides
    dro = (ChiSqDRO(
        n_train=len(train_ds_full),
        alpha=alpha,
        eta=CFG["dro_eta"],
        lambda_init=CFG["dro_lambda_init"],
        warmup_epochs=CFG["dro_warmup"],
        w_clip=_ACTIVE_OVERRIDES.get("w_clip", (0.2, 5.0)),
        max_ratio=_ACTIVE_OVERRIDES.get("max_ratio", 10.0),
        uniform_mix=_ACTIVE_OVERRIDES.get("uniform_mix", 0.5),
    ) if is_dro else None)

    ckpt = load_checkpoint(cell_dir)
    # Sanity check: discard stale checkpoint if canary count changed (e.g., k override)
    expected_k = len(canary_ds.labels)
    if ckpt is not None:
        ckpt_init_losses = ckpt["extra"].get("initial_losses")
        if ckpt_init_losses is not None and len(ckpt_init_losses) != expected_k:
            print(f"⚠️  Stale checkpoint (init_losses len={len(ckpt_init_losses)} ≠ k={expected_k}) — discarding")
            for f in cell_dir.glob("*.pt"):
                f.unlink()
            ckpt = None
    resume_epoch = ckpt["epoch"] if ckpt else 0
    init_params  = ckpt["extra"]["initial_params"]  if ckpt else None
    init_losses  = ckpt["extra"]["initial_losses"]  if ckpt else None
    if ckpt and is_dro and ckpt.get("dro_state"):
        dro.load_state(ckpt["dro_state"])

    print(f"\n{'═'*64}\n▶ {eps_label} / {tag.upper()} "
          f"(attempt {attempt}, σ={sigma}, α={alpha:.6g})\n{'═'*64}")
    print(f"   overrides: {_ACTIVE_OVERRIDES or '(defaults)'}")

    run = train_one_run(is_dro, sigma, dro, CFG["epochs"], cell_dir,
                        resume_epoch=resume_epoch,
                        initial_params=init_params,
                        initial_losses=init_losses)

    bb, wb, preclip = compute_attack_scores(run)
    bb_audit = steinke_audit(bb, membership.numpy())
    wb_audit = steinke_audit(wb, membership.numpy())
    subgroups = subgroup_analysis(wb, preclip, membership.numpy(), canary_subgroups)

    result = {
        "dataset": DATASET, "eps_label": eps_label, "method": tag,
        "attempt": attempt, "sigma": sigma, "alpha_used": alpha,
        "overrides_applied": dict(_ACTIVE_OVERRIDES),
        "epsilon_spent": run["epsilon_spent"],
        "history": run["history"],
        "final_metrics": {k: v for k, v in run["final_metrics"].items() if k != "race_err"},
        "race_err": run["final_metrics"]["race_err"],
        "bb_audit":  bb_audit,  "wb_audit": wb_audit,
        "subgroup_auc": subgroups,
        "scores": {"bb": bb.tolist(), "wb": wb.tolist(),
                   "preclip_norms": preclip.tolist()},
        "dro_state_final": ({"lambda_t": dro.lambda_t,
                              "activation_ratio": dro.activation_ratio,
                              "lambda_max": dro.lambda_max}
                             if is_dro else None),
        "timestamp": datetime.now().isoformat(),
    }

    if is_dro:
        ok, reason, new_alpha = validate_dro_result(result, alpha)
        result["validated"]     = ok
        result["invalid_reason"] = reason if not ok else None
        if not ok and attempt < 3 and new_alpha is not None \
                and new_alpha not in tried_alphas:
            print(f"⚠️ DRO invalid ({reason}) → retrying with α={new_alpha:.6g}")
            save_json_atomic(results_path, result)
            for f in list(cell_dir.glob("*.pt")): f.unlink()
            return run_cell(eps_label, sigma, True, new_alpha,
                            attempt + 1, tried_alphas + [alpha, new_alpha])
    else:
        result["validated"] = True

    save_json_atomic(results_path, result)
    print(f"💾 Saved → {results_path}")
    return result


In [16]:
# ══════════════════════════════════════════════════════════════════════
# QUALITY RUBRIC — per-(ε, method-pair) pass/fail assessment.
# Used by the auto-tuner to decide whether to suggest adjustments.
# ══════════════════════════════════════════════════════════════════════
THRESHOLDS = {
    "c1_race_gap_reduction": 0.10,   # DRO must reduce race_gap by ≥10% vs ERM
    "c2_wb_auc_increase":    0.02,   # DRO WB AUC must exceed ERM WB AUC by ≥0.02
    "c3_dispersion_increase": 0.03,  # DRO race disparity must exceed ERM by ≥0.03
    "dro_active_min":        0.10,   # λ/max must be ≥10% for DRO to be "active"
    "dro_active_max":        0.90,   # λ/max must be ≤90% to avoid saturation
    "signal_auc":            0.55,   # at least one AUC ≥0.55 OR max subgroup ≥0.58
    "signal_sub_auc":        0.58,
    "subgroup_min_n":        30,     # every reported subgroup needs ≥30 in & ≥30 out
}

def _race_disparity(subgroup_data):
    """Max−min AUC across race subgroups (excluding aggregate keys)."""
    aucs = [v["auc"] for k, v in subgroup_data.get("race", {}).items()
            if not k.startswith("_") and isinstance(v, dict)]
    return (max(aucs) - min(aucs)) if len(aucs) > 1 else 0.0

def _max_subgroup_auc(subgroup_data):
    aucs = [v["auc"] for k, v in subgroup_data.get("race", {}).items()
            if not k.startswith("_") and isinstance(v, dict)]
    return max(aucs) if aucs else 0.5

def _min_subgroup_n(subgroup_data):
    ns = [min(v["n_in"], v["n_out"]) for k, v in subgroup_data.get("race", {}).items()
          if not k.startswith("_") and isinstance(v, dict)]
    return min(ns) if ns else 0

def evaluate_run_quality(erm_result, dro_result, eps_label):
    """Evaluate whether the (ERM, DRO) pair at this ε meets the quality bar.

    Returns dict with per-claim booleans and underlying metrics.
    """
    m = {}
    m["eps_label"] = eps_label   # ← ADD THIS LINE
    m["alpha_used"] = float(dro_result.get("alpha_used", CFG.get("dro_alpha_frac",0.7)))
    m["race_gap_erm"] = erm_result["final_metrics"]["race_gap"]
    m["race_gap_dro"] = dro_result["final_metrics"]["race_gap"]
    m["wb_auc_erm"]   = erm_result["wb_audit"]["auc"]
    m["wb_auc_dro"]   = dro_result["wb_audit"]["auc"]
    m["bb_auc_erm"]   = erm_result["bb_audit"]["auc"]
    m["bb_auc_dro"]   = dro_result["bb_audit"]["auc"]
    m["erm_disp"]     = _race_disparity(erm_result["subgroup_auc"])
    m["dro_disp"]     = _race_disparity(dro_result["subgroup_auc"])
    m["max_sub_auc"]  = _max_subgroup_auc(dro_result["subgroup_auc"])
    m["n_min"]        = _min_subgroup_n(dro_result["subgroup_auc"])
    nu_traj = dro_result["history"].get("activation_ratio", [])
    m["lam_ratio"]       = max(nu_traj) if nu_traj else 0.0    # ← was dro_state_final
    m["lam_ratio_final"] = nu_traj[-1] if nu_traj else 0.0
    m["race_gap_prob_erm"] = erm_result["final_metrics"].get("race_gap_prob", 0.0)
    m["race_gap_prob_dro"] = dro_result["final_metrics"].get("race_gap_prob", 0.0)
    m["pred_std_dro"]      = dro_result["history"].get("pred_std", [0.0])[-1]

    def _red(a, b):
        return (a - b) / a if a > 1e-6 else 0.0

    red_err  = _red(m["race_gap_erm"],  m["race_gap_dro"])
    red_prob = _red(m["race_gap_prob_erm"], m["race_gap_prob_dro"])
    m["c1_reduction"] = max(red_err, red_prob)

    # C1: fairness — accept improvement in EITHER error-rate or probability space
    c1_pass = m["c1_reduction"] >= THRESHOLDS["c1_race_gap_reduction"]

    # C2 / C3 are only physically attainable where DP noise has not destroyed
    # the gradient signal. Requiring them at ε≤4 makes the tuner chase noise.
    strict = eps_label in ("no_dp", "eps8")

    # # C1: fairness — DRO must reduce race_gap by ≥ THRESHOLDS["c1_race_gap_reduction"]
    # # (only meaningful if ERM actually has a non-trivial gap to begin with)
    # c1_pass = (m["race_gap_erm"] > 1e-4 and
    #            m["race_gap_dro"] < m["race_gap_erm"] * (1 - THRESHOLDS["c1_race_gap_reduction"]))

    # C2: privacy leakage — DRO WB AUC > ERM WB AUC by ≥ THRESHOLDS["c2_wb_auc_increase"]
    c2_pass = m["wb_auc_dro"] > m["wb_auc_erm"] + THRESHOLDS["c2_wb_auc_increase"]

    # C3: disparate vulnerability — DRO race disparity > ERM race disparity
    c3_pass = m["dro_disp"] > m["erm_disp"] + THRESHOLDS["c3_dispersion_increase"]

    # DRO activation: λ/max ∈ [0.10, 0.90]
    dro_active = THRESHOLDS["dro_active_min"] <= m["lam_ratio"] <= THRESHOLDS["dro_active_max"]

    # Signal presence: at least one AUC ≥ 0.55 OR max subgroup ≥ 0.58
    has_signal = (m["wb_auc_dro"] >= THRESHOLDS["signal_auc"] or
                  m["wb_auc_erm"] >= THRESHOLDS["signal_auc"] or
                  m["max_sub_auc"] >= THRESHOLDS["signal_sub_auc"])

    # Subgroup sample adequacy
    n_ok = m["n_min"] >= THRESHOLDS["subgroup_min_n"]

    # Trivial collapse
    trivial = dro_result["history"].get("trivial_collapse", False)

    # Utility diverging (last utility > 1.6× min over history)
    utils = dro_result["history"]["utility"]
    util_diverging = (len(utils) > 0 and utils[-1] > 1.6 * min(utils)) if utils else False

    overall_pass = (c1_pass and dro_active and n_ok and not trivial
                    and ((c2_pass and c3_pass) if strict else True))

    return {
        "eps_label": eps_label, "strict": strict,
        "c1_pass": c1_pass, "c2_pass": c2_pass, "c3_pass": c3_pass,
        "dro_active": dro_active, "has_signal": has_signal, "n_ok": n_ok,
        "trivial_collapse": trivial, "utility_diverging": util_diverging,
        "lam_ratio": m["lam_ratio"], "overall_pass": overall_pass,
        "metrics": m,
    }


In [17]:
# ══════════════════════════════════════════════════════════════════════
# PARAMETER ADJUSTER — maps quality failures to PER-ε parameter overrides.
# Seven rules. α and lr are scoped per-ε; k, max_ratio, uniform_mix are global.
# ══════════════════════════════════════════════════════════════════════

def _as_float(d, key, default):
    try:    return float(d.get(key, default))
    except (TypeError, ValueError): return float(default)

def _as_int(d, key, default):
    try:    return int(d.get(key, default))
    except (TypeError, ValueError): return int(default)


def suggest_param_adjustments(quality, current_overrides):
    """Given a quality assessment for ONE ε cell, return (suggestions, reasons).

    Per-ε keys (dicts keyed by eps_label):
        alpha_per_eps, lr_per_eps, warmup_per_eps
    Global keys (scalars):
        k, max_ratio, uniform_mix, dro_eta

    Rules:
      1.  λ/max < 10%                         → α too high for this ε  → α × 0.4  (per-ε)
      2.  λ/max > 90% + utility diverging     → α too low for this ε   → α × 2.0  (per-ε)
      2b. λ/max > 90% + C1 fails + util ok    → α too low for this ε   → α × 2.0  (per-ε)
      3.  DRO active but C1 fails             → weights too uniform    → max_ratio×2, uniform_mix×0.5
      4.  DRO active + no signal + k < cap    → k too small            → k × 2
      5.  n_min < 30                          → k too small            → k × 2
      6.  trivial_collapse                    → lr too high for this ε → lr×0.25, warmup+3 (per-ε)
    """
    suggestions = {}
    reasons = []
    m = quality["metrics"]
    eps_label = quality.get("eps_label", "unknown")   # ← FIX: safe extraction

    # Helper: read current per-ε α (from overrides, then from result, then from CFG)
    def _cur_alpha():
        ape = current_overrides.get("alpha_per_eps", {})
        if eps_label in ape:
            return float(ape[eps_label])
        return _as_float(m, "alpha_used",
                         CFG.get("dro_alpha_init", CFG.get("dro_alpha_frac", 1.0)))

    # Helper: read current per-ε lr
    def _cur_lr():
        lpe = current_overrides.get("lr_per_eps", {})
        if eps_label in lpe:
            return float(lpe[eps_label])
        return _as_float(current_overrides, "lr", CFG["lr"])

    # Helper: read current per-ε warmup
    def _cur_warmup():
        wpe = current_overrides.get("warmup_per_eps", {})
        if eps_label in wpe:
            return int(wpe[eps_label])
        return _as_int(current_overrides, "dro_warmup", CFG["dro_warmup"])

    # ── Rule 1: DRO dormant (λ/max < 10%) → α too high for this ε ──
    if m["lam_ratio"] < THRESHOLDS["dro_active_min"]:
        cur_a = _cur_alpha()
        new_a = max(cur_a / 2.0, ALPHA_FLOOR)
        suggestions.setdefault("alpha_per_eps", {})[eps_label] = new_a
        reasons.append(f"[{eps_label}] λ/max={m['lam_ratio']:.1%} <10% (dormant) "
                       f"→ α {cur_a:.4g}→{new_a:.4g}")

    # ── Rule 2: saturated + utility diverging → α too low ──
    elif m["lam_ratio"] > THRESHOLDS["dro_active_max"] and quality["utility_diverging"]:
        cur_a = _cur_alpha()
        new_a = min(cur_a * MAX_ALPHA_RATIO, cur_a * 2.0)
        suggestions.setdefault("alpha_per_eps", {})[eps_label] = new_a
        reasons.append(f"[{eps_label}] λ/max={m['lam_ratio']:.0%} saturated + util diverging "
                       f"→ α {cur_a:.4g}→{new_a:.4g}")

    # ── Rule 2b (NEW): saturated + C1 fails + utility NOT diverging → α too low ──
    elif (m["lam_ratio"] > THRESHOLDS["dro_active_max"]
          and not quality["c1_pass"]
          and not quality["utility_diverging"]):
        cur_a = _cur_alpha()
        new_a = min(cur_a * MAX_ALPHA_RATIO, cur_a * 2.0)
        suggestions.setdefault("alpha_per_eps", {})[eps_label] = new_a
        reasons.append(f"[{eps_label}] λ/max={m['lam_ratio']:.0%} saturated + C1 fails "
                       f"(gap {m.get('race_gap_erm',0):.3f}→{m.get('race_gap_dro',0):.3f}) "
                       f"→ α {cur_a:.4g}→{new_a:.4g}")

    # ── Rule 3: DRO active but C1 fails → trust region too tight ──
    if quality["dro_active"] and not quality["c1_pass"]:
        cur_mr = _as_float(current_overrides, "max_ratio", 10.0)
        cur_um = _as_float(current_overrides, "uniform_mix", 0.5)
        suggestions["max_ratio"]   = min(cur_mr * 2, 50.0)
        suggestions["uniform_mix"] = max(cur_um * 0.5, 0.1)
        reasons.append(f"[{eps_label}] DRO active (λ={m['lam_ratio']:.0%}) but C1 fails "
                       f"(reduction {m.get('c1_reduction',0):.1%}) → "
                       f"max_ratio {cur_mr:g}→{min(cur_mr*2,50.0):g}, "
                       f"uniform_mix {cur_um:g}→{max(cur_um*0.5,0.1):g}")

    # ── Rules 4/5: k too small ──
    if (quality["dro_active"] and not quality["has_signal"]) or (not quality["n_ok"]):
        cur_k = _as_int(current_overrides, "k", CFG["k"])
        if cur_k < MAX_K_HARD_CAP and "k" not in suggestions:
            suggestions["k"] = min(cur_k * 2, MAX_K_HARD_CAP)
            reasons.append(f"[{eps_label}] signal/n inadequate "
                           f"(max_sub={m['max_sub_auc']:.3f}, n_min={m['n_min']}) "
                           f"→ k {cur_k}→{suggestions['k']}")

    # ── Rule 6: trivial collapse → lr too high FOR THIS ε ──
    if quality["trivial_collapse"]:
        cur_lr = _cur_lr()
        cur_wu = _cur_warmup()
        new_lr = max(cur_lr / MAX_LR_RATIO, 1e-6)
        new_wu = min(cur_wu + 3, MAX_WARMUP)
        suggestions.setdefault("lr_per_eps", {})[eps_label] = new_lr
        suggestions.setdefault("warmup_per_eps", {})[eps_label] = new_wu
        reasons.append(f"[{eps_label}] Trivial collapse "
                       f"(pred_std={m.get('pred_std_dro',0):.4f}) → "
                       f"lr {cur_lr:g}→{new_lr:g}, warmup {cur_wu}→{new_wu}")

    if not reasons:
        reasons.append(f"[{eps_label}] No adjustments needed — all checks pass")
    return suggestions, reasons

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# AUTO-TUNE DRIVER — per-ε overrides, fail-proof merge, smart artifact clearing
# ══════════════════════════════════════════════════════════════════════

# Keys whose values are dicts keyed by eps_label (merged as dicts, not scalars)
_PER_EPS_KEYS = {"alpha_per_eps", "lr_per_eps", "warmup_per_eps"}


def apply_overrides_to_cfg(overrides):
    """Apply GLOBAL overrides to CFG in-place. Per-ε keys are NOT applied here."""
    if not overrides:
        return
    if "dro_eta" in overrides:  CFG["dro_eta"] = float(overrides["dro_eta"])
    # lr, dro_warmup are now per-ε — do NOT mutate CFG globally


def apply_per_eps_to_cfg(overrides, eps_label):
    """Temporarily apply per-ε overrides to CFG. Returns saved values for restoration."""
    saved = {}
    lr_map = overrides.get("lr_per_eps", {})
    wu_map = overrides.get("warmup_per_eps", {})
    if eps_label in lr_map:
        saved["lr"] = CFG["lr"]; CFG["lr"] = float(lr_map[eps_label])
    if eps_label in wu_map:
        saved["dro_warmup"] = CFG["dro_warmup"]; CFG["dro_warmup"] = int(wu_map[eps_label])
    return saved


def restore_cfg(saved):
    """Restore CFG values saved by apply_per_eps_to_cfg."""
    CFG.update(saved)


def clear_run_artifacts(eps_labels, methods=("erm", "dro")):
    """Delete result.json and checkpoints for specified ε labels and methods."""
    for eps_label in eps_labels:
        for tag in methods:
            d = EXP_ROOT / eps_label / tag
            if d.exists():
                for f in d.glob("*.pt"): f.unlink()
                rj = d / "result.json"
                if rj.exists(): rj.unlink()


def _merge_suggestions(a, b):
    """Merge suggestion dicts. Per-ε keys merge as dicts; scalars use geometric mean for α."""
    merged = dict(a); conflicts = []
    for k, v in b.items():
        if k in _PER_EPS_KEYS:
            # Dict merge: per-ε values from different cells don't conflict
            if k not in merged:
                merged[k] = dict(v)
            else:
                merged[k] = {**merged[k], **v}
        elif k not in merged:
            merged[k] = v
        elif merged[k] == v:
            continue
        elif k == "alpha_override":
            # Legacy global α (shouldn't happen with per-ε, but handle gracefully)
            merged[k] = float(np.sqrt(merged[k] * v)); conflicts.append(k)
        else:
            merged[k] = v   # last-write-wins for bounded scalars (k, max_ratio, etc.)
    return merged, conflicts


def auto_tune_sweep(max_rounds=None):
    global _ACTIVE_OVERRIDES, all_results, tune_history
    _ACTIVE_OVERRIDES = {}
    tune_history = []
    if max_rounds is None: max_rounds = MAX_TUNING_ROUNDS

    # Snapshot original CFG so we can restore between rounds
    _CFG_ORIGINAL = {k: v for k, v in CFG.items()
                     if isinstance(v, (int, float, str, bool, type(None)))}

    print(f"\n{'#'*80}\n# AUTO-TUNE SWEEP — max_rounds={max_rounds}\n{'#'*80}")

    for round_num in range(1, max_rounds + 1):
        round_t0 = time.time()
        print(f"\n{'#'*80}\n# ROUND {round_num}/{max_rounds}\n{'#'*80}")
        print(f"   Global overrides: "
              f"{ {k:v for k,v in _ACTIVE_OVERRIDES.items() if k not in _PER_EPS_KEYS} or '(none)'}")
        for pk in _PER_EPS_KEYS:
            if pk in _ACTIVE_OVERRIDES:
                print(f"   {pk}: {_ACTIVE_OVERRIDES[pk]}")

        # Restore CFG to original, then apply global overrides
        CFG.update(_CFG_ORIGINAL)
        apply_overrides_to_cfg(_ACTIVE_OVERRIDES)

        # ── Rebuild split if k changed ──
        prev_k = (tune_history[-1]["overrides_applied"].get("k", CFG.get("k", 500))
                  if tune_history else CFG.get("k", 500))
        cur_k  = _ACTIVE_OVERRIDES.get("k", CFG.get("k", 500))
        if cur_k != prev_k:
            print(f"⚠️ k changed {prev_k}→{cur_k} — rebuilding split & datasets")
            construct_split(k_override=cur_k)
            build_phase_datasets()
            if CALIB_PATH.exists(): CALIB_PATH.unlink()
            clear_run_artifacts([("no_dp" if e is None else f"eps{int(e)}")
                                 for e in EPSILON_SWEEP])

        # ── Run the sweep with PER-ε α calibration ──
        all_results = {}
        alpha_per_eps = {}
        alpha_overrides = _ACTIVE_OVERRIDES.get("alpha_per_eps", {})

        for eps in EPSILON_SWEEP:
            if eps is None: eps_label, sigma = "no_dp", None
            else:           eps_label, sigma = f"eps{int(eps)}", CFG["sigma_map"][eps]

            # Apply per-ε lr/warmup to CFG temporarily
            saved_cfg = apply_per_eps_to_cfg(_ACTIVE_OVERRIDES, eps_label)

            try:
                # Step 1: Run ERM (α irrelevant)
                erm_result = run_cell(eps_label, sigma, False, 0.0)
                all_results[(eps_label, "erm")] = erm_result

                # Step 2: Calibrate α from THIS ε's converged ERM variance
                if eps_label in alpha_overrides:
                    alpha_eps = float(alpha_overrides[eps_label])
                    print(f"   [{eps_label}] α overridden (per-ε): {alpha_eps:.6g}")
                elif "alpha_override" in _ACTIVE_OVERRIDES:
                    # Legacy global fallback
                    alpha_eps = float(_ACTIVE_OVERRIDES["alpha_override"])
                    print(f"   [{eps_label}] α overridden (global): {alpha_eps:.6g}")
                else:
                    alpha_eps, V_conv = calibrate_alpha_from_erm(erm_result)
                    print(f"   [{eps_label}] α from converged ERM: "
                          f"α={alpha_eps:.6g} (V_conv={V_conv:.4g})")
                alpha_per_eps[eps_label] = alpha_eps

                # Step 3: Run DRO with ε-specific α
                dro_result = run_cell(eps_label, sigma, True, alpha_eps)
                all_results[(eps_label, "dro")] = dro_result

            finally:
                restore_cfg(saved_cfg)

        # ── Evaluate quality per ε ──
        round_assessments = {}; any_unsatisfactory = False
        for eps in EPSILON_SWEEP:
            eps_label = "no_dp" if eps is None else f"eps{int(eps)}"
            quality = evaluate_run_quality(all_results[(eps_label, "erm")],
                                           all_results[(eps_label, "dro")], eps_label)
            # Ensure eps_label is in the quality dict for suggest_param_adjustments
            quality["eps_label"] = eps_label
            round_assessments[eps_label] = quality
            print(f"  {eps_label}: {'✅' if quality['overall_pass'] else '❌'} "
                  f"C1={quality['c1_pass']}, C2={quality['c2_pass']}, C3={quality['c3_pass']}, "
                  f"λ={quality['lam_ratio']:.0%}, signal={quality['has_signal']}, "
                  f"n_ok={quality['n_ok']}")
            if not quality["overall_pass"]: any_unsatisfactory = True

        round_dt = time.time() - round_t0
        tune_history.append({
            "round": round_num,
            "overrides_applied": dict(_ACTIVE_OVERRIDES),
            "alpha_per_eps": alpha_per_eps,
            "duration_sec": round_dt,
            "assessments": round_assessments,
        })
        print(f"   Round {round_num} duration: {round_dt:.1f}s")

        if not any_unsatisfactory:
            print(f"\n🎉 Round {round_num}: ALL ε CELLS PASS — stopping early"); break

        # ── Collect adjustments from ALL failing cells ──
        new_suggestions = dict(_ACTIVE_OVERRIDES); all_reasons = []
        for eps_label, q in round_assessments.items():
            if q["overall_pass"]: continue
            sugg, reasons = suggest_param_adjustments(q, new_suggestions)
            all_reasons += reasons
            new_suggestions, _conflicts = _merge_suggestions(new_suggestions, sugg)

        # Filter out no-op suggestions
        new_suggestions = {k: v for k, v in new_suggestions.items()
                           if not (isinstance(v, (int, float))
                                   and isinstance(_ACTIVE_OVERRIDES.get(k), (int, float, type(None)))
                                   and _ACTIVE_OVERRIDES.get(k) == v)}
        # Also filter per-ε dicts that haven't changed
        for pk in _PER_EPS_KEYS:
            if pk in new_suggestions and pk in _ACTIVE_OVERRIDES:
                if new_suggestions[pk] == _ACTIVE_OVERRIDES[pk]:
                    del new_suggestions[pk]

        if not new_suggestions or new_suggestions == _ACTIVE_OVERRIDES:
            print(f"\n⛔ Round {round_num}: no new adjustments — converged"); break

        print(f"\n📋 Adjustments for round {round_num+1}:")
        for r in all_reasons: print(f"   - {r}")
        print(f"   → merged override set: {new_suggestions}")
        _ACTIVE_OVERRIDES = new_suggestions   # ← FIX: replace, don't update (avoids stale keys)

        # ── Smart artifact clearing ──
        # Only clear DRO for α changes; clear both ERM+DRO if lr changed for that ε
        lr_changed_eps = set(_ACTIVE_OVERRIDES.get("lr_per_eps", {}).keys())
        dro_only_eps = []; both_eps = []
        for eps_label, q in round_assessments.items():
            if q["overall_pass"]: continue
            if eps_label in lr_changed_eps:
                both_eps.append(eps_label)     # lr changed → ERM must re-run too
            else:
                dro_only_eps.append(eps_label) # only α/trust-region changed → DRO only

        if dro_only_eps:
            clear_run_artifacts(dro_only_eps, methods=("dro",))
            print(f"   Cleared DRO artifacts for: {dro_only_eps}")
        if both_eps:
            clear_run_artifacts(both_eps, methods=("erm", "dro"))
            print(f"   Cleared ERM+DRO artifacts for: {both_eps} (lr changed)")

    # Restore CFG to original after all rounds
    CFG.update(_CFG_ORIGINAL)

    save_json_atomic(EXP_ROOT / "tune_history.json", tune_history)
    print(f"\n💾 Tuning history → {EXP_ROOT}/tune_history.json")
    return all_results, tune_history


all_results, tune_history = auto_tune_sweep()
print("\n✅ Auto-tune sweep complete")


################################################################################
# AUTO-TUNE SWEEP — max_rounds=3
################################################################################

################################################################################
# ROUND 1/3
################################################################################
   Global overrides: (none)

════════════════════════════════════════════════════════════════
▶ no_dp / ERM (attempt 1, σ=None, α=0)
════════════════════════════════════════════════════════════════
   overrides: (defaults)
  [ERM σ=None] Ep  5 | L=0.310 | U=0.147 | ε=0.00 | ν=0% | V=0.2503 | 3s
  [ERM σ=None] Ep 10 | L=0.290 | U=0.143 | ε=0.00 | ν=0% | V=0.238 | 7s
  [ERM σ=None] Ep 15 | L=0.272 | U=0.156 | ε=0.00 | ν=0% | V=0.2282 | 10s
  [ERM σ=None] Ep 20 | L=0.257 | U=0.154 | ε=0.00 | ν=0% | V=0.187 | 13s
  ⚠️ [ERM] Early stop at epoch 25.
💾 Saved → /home/govindu.srimaan/project/monolith/experiments/adult_eps_sweep/no

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# AGGREGATION — paired stats (ERM vs DRO per ε) + paper-ready tables
# ══════════════════════════════════════════════════════════════════════
summary_rows = []
for eps in EPSILON_SWEEP:
    eps_label = "no_dp" if eps is None else f"eps{int(eps)}"
    erm = all_results.get((eps_label, "erm")); dro = all_results.get((eps_label, "dro"))
    if not erm or not dro: continue

    bb_paired = paired_tests(np.array(erm["scores"]["bb"]),  np.array(dro["scores"]["bb"]))
    wb_paired = paired_tests(np.array(erm["scores"]["wb"]),  np.array(dro["scores"]["wb"]))
    gn_paired = paired_tests(np.array(erm["scores"]["preclip_norms"]),
                             np.array(dro["scores"]["preclip_norms"]))

    row = {
        "epsilon": eps_label, "eps_realised_erm": erm["epsilon_spent"],
        "eps_realised_dro": dro["epsilon_spent"],
        "bb_auc_erm": erm["bb_audit"]["auc"], "bb_auc_dro": dro["bb_audit"]["auc"],
        "bb_delta":   dro["bb_audit"]["auc"] - erm["bb_audit"]["auc"],
        "bb_paired_p": bb_paired["wilcoxon_p"], "bb_cohens_d": bb_paired["cohens_d_paired"],
        "wb_auc_erm": erm["wb_audit"]["auc"], "wb_auc_dro": dro["wb_audit"]["auc"],
        "wb_delta":   dro["wb_audit"]["auc"] - erm["wb_audit"]["auc"],
        "wb_paired_p": wb_paired["wilcoxon_p"], "wb_cohens_d": wb_paired["cohens_d_paired"],
        "gn_erm": float(np.mean(erm["scores"]["preclip_norms"])),
        "gn_dro": float(np.mean(dro["scores"]["preclip_norms"])),
        "gn_paired_p": gn_paired["wilcoxon_p"],
        "utility_erm": erm["final_metrics"]["utility"], "utility_dro": dro["final_metrics"]["utility"],
        "race_gap_erm": erm["final_metrics"]["race_gap"],   "race_gap_dro": dro["final_metrics"]["race_gap"],
        "gender_gap_erm": erm["final_metrics"]["gender_gap"], "gender_gap_dro": dro["final_metrics"]["gender_gap"],
        "lambda_final": dro["dro_state_final"]["lambda_t"] if dro["dro_state_final"] else 0.0,
        "activation_ratio": dro["dro_state_final"]["activation_ratio"] if dro["dro_state_final"] else 0.0,
        "dro_validated": dro["validated"],
    }
    summary_rows.append(row)

save_json_atomic(EXP_ROOT / "sweep_summary.json",
                 {"dataset": DATASET, "rows": summary_rows, "timestamp": datetime.now().isoformat()})

print(f"\n{'═'*118}\n  {DATASET.upper()} — ε SWEEP SUMMARY (post-tuning)\n{'═'*118}")
hdr = f"{'ε':<7}{'BB AUC E→D':<16}{'Δ':<8}{'p':<9}{'WB AUC E→D':<16}{'Δ':<8}{'p':<9}{'‖g‖ E→D':<16}{'λ/max':<8}{'RaceGap E→D':<16}"
print(hdr); print("─"*118)
for r in summary_rows:
    print(f"{r['epsilon']:<7}"
          f"{r['bb_auc_erm']:.3f}→{r['bb_auc_dro']:.3f}   {r['bb_delta']:+.3f}  {r['bb_paired_p']:<8.4f}"
          f"{r['wb_auc_erm']:.3f}→{r['wb_auc_dro']:.3f}   {r['wb_delta']:+.3f}  {r['wb_paired_p']:<8.4f}"
          f"{r['gn_erm']:.2f}→{r['gn_dro']:.2f}      {r['activation_ratio']*100:>5.1f}%  "
          f"{r['race_gap_erm']:.3f}→{r['race_gap_dro']:.3f}")
print("═"*118)


In [ ]:
TARGET_EPS = "eps8" if any(r["epsilon"] == "eps8" for r in summary_rows) else summary_rows[0]["epsilon"]
erm_r = all_results[(TARGET_EPS, "erm")]
dro_r = all_results[(TARGET_EPS, "dro")]

print(f"\n{'═'*96}\n  SUBGROUP DISPARATE VULNERABILITY — {DATASET.upper()} @ {TARGET_EPS} (White-Box)\n{'═'*96}")
print(f"{'Subgroup':<12}{'n_in':<7}{'n_out':<7}{'ERM AUC':<10}{'DRO AUC':<10}{'Δ':<9}{'‖g‖ ERM':<10}{'‖g‖ DRO':<10}")
print("─"*96)

for axis in ("race", "gender"):
    for gname in dro_r["subgroup_auc"][axis]:
        if gname.startswith("_"):
            continue
        d = dro_r["subgroup_auc"][axis][gname]
        if not isinstance(d, dict):
            continue
        e = erm_r["subgroup_auc"][axis].get(gname, {})
        e_auc = e.get("auc", float("nan"))                if isinstance(e, dict) else float("nan")
        e_gn  = e.get("preclip_grad_norm", float("nan"))  if isinstance(e, dict) else float("nan")
        delta = d["auc"] - e_auc
        print(f"{gname:<12}{d['n_in']:<7}{d['n_out']:<7}{e_auc:<10.3f}{d['auc']:<10.3f}"
              f"{delta:<+9.3f}{e_gn:<10.3f}{d['preclip_grad_norm']:<10.3f}")
    disp = dro_r["subgroup_auc"][axis].get("_disparity", 0.0)
    print(f"  → {axis} disparity (DRO): {disp:.3f}")

print("═"*96)
save_json_atomic(EXP_ROOT / f"subgroup_table_{TARGET_EPS}.json",
                 {"erm": erm_r["subgroup_auc"], "dro": dro_r["subgroup_auc"]})
print(f"💾 Saved → {EXP_ROOT}/subgroup_table_{TARGET_EPS}.json")


In [ ]:
print(f"""
{'═'*70}
  ✅ {DATASET.upper()} SWEEP COMPLETE
{'═'*70}
Results:      {EXP_ROOT}
Summary JSON: {EXP_ROOT}/sweep_summary.json
Tune history: {EXP_ROOT}/tune_history.json

VALIDATION STATUS:""")
print("\nKEY READOUTS (data-driven):")
for r in summary_rows:
    bb_sig = "✓" if (r["bb_delta"] > 0 and r["bb_paired_p"] < 0.05) else "✗"
    wb_sig = "✓" if (r["wb_delta"] > 0 and r["wb_paired_p"] < 0.05) else "✗"
    gn_up  = "✓" if r["gn_dro"] > r["gn_erm"] else "✗"
    fair   = "✓" if r["race_gap_dro"] < r["race_gap_erm"] - 1e-4 else "✗"
    print(f"  ε={r['epsilon']:<7} BBΔ{r['bb_delta']:+.3f}{bb_sig}  "
          f"WBΔ{r['wb_delta']:+.3f}{wb_sig}  ‖g‖{gn_up}  FairnessGap↓{fair}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TUNING HISTORY — what was tried, what changed, what improved
# ══════════════════════════════════════════════════════════════════════
from collections import Counter

print(f"\n{'═'*96}\n  TUNING HISTORY — {DATASET.upper()}\n{'═'*96}")
print(f"{'Round':<6}{'Duration':<10}{'Overrides':<55}{'ε pass rate':<15}")
print("─"*96)
for h in tune_history:
    n_eps = len(h["assessments"])
    n_pass = sum(1 for q in h["assessments"].values() if q["overall_pass"])
    ovr = h["overrides_applied"]
    ovr_str = ", ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}"
                         for k, v in ovr.items()) or "(defaults)"
    if len(ovr_str) > 53: ovr_str = ovr_str[:50] + "..."
    print(f"{h['round']:<6}{h['duration_sec']:<10.1f}{ovr_str:<55}{n_pass}/{n_eps:<13}")
print("═"*96)

# Per-round quality trajectory
print(f"\n  PER-ROUND QUALITY TRAJECTORY")
for h in tune_history:
    _alpha = h.get('alpha_used')
    if _alpha is None:
        _ape = h.get('alpha_per_eps', {})
        _alpha = ", ".join(f"{k}:{v:.4g}" for k, v in _ape.items()) if _ape else "n/a"
        print(f"\n  Round {h['round']} (α per-ε: {_alpha}):")
    else:
        print(f"\n  Round {h['round']} (α={_alpha:.4g}):")
  
    for eps_label, q in h["assessments"].items():
        status = "✅" if q["overall_pass"] else "❌"
        m = q["metrics"]
        print(f"    {eps_label}: {status}  "
              f"C1(race_gap {m['race_gap_erm']:.3f}→{m['race_gap_dro']:.3f}) "
              f"C2(WB {m['wb_auc_erm']:.3f}→{m['wb_auc_dro']:.3f}) "
              f"C3(disp {m['erm_disp']:.3f}→{m['dro_disp']:.3f}) "
              f"λ/max={q['lam_ratio']:.0%}")

# Final recommendations
print(f"\n{'═'*96}\n  RECOMMENDATIONS\n{'═'*96}")
last = tune_history[-1] if tune_history else None
if last:
    final_pass = sum(1 for q in last["assessments"].values() if q["overall_pass"])
    n_eps = len(last["assessments"])
    if final_pass == n_eps:
        print(f"  ✅ All {n_eps} ε cells pass the quality bar. Results are paper-ready.")
    elif final_pass > 0:
        print(f"  ⚠️  {final_pass}/{n_eps} ε cells pass. Consider:")
        print(f"     - Increase MAX_TUNING_ROUNDS to allow more iterations.")
        print(f"     - Run on a stronger dataset (UTKFace) for larger gradient signal.")
        print(f"     - Run multiple seeds for paired-stat stability (VALIDITY-03).")
    else:
        print(f"  ❌ No ε cells pass. Likely root causes:")
        # Identify the dominant failure mode
        all_failures = []
        for q in last["assessments"].values():
            if q["lam_ratio"] < THRESHOLDS["dro_active_min"]:
                all_failures.append("dro_dormant")
            elif q["lam_ratio"] > THRESHOLDS["dro_active_max"]:
                all_failures.append("dro_saturated")
            if not q["c1_pass"]:    all_failures.append("c1_fail")
            if not q["c2_pass"]:    all_failures.append("c2_fail")
            if not q["has_signal"]: all_failures.append("no_signal")
            if not q["n_ok"]:       all_failures.append("small_n")
        for cause, n in Counter(all_failures).most_common():
            print(f"     - {cause}: {n} ε cells")
        print(f"\n  Suggested next steps:")
        print(f"     - Run on UTKFace (ResNet-18 has 11M params → much stronger gradient signal).")
        print(f"     - Manually tune σ_map if ε-budget allows lower noise.")
        print(f"     - If on tabular only, consider a stronger model (MLP instead of LogReg).")
print("═"*96)
